In [ ]:
import pandas as pd
from pathlib import Path
from numpy.linalg import norm
from scipy.stats import percentileofscore

# Rutas base
base_raw = Path('data/raw')
base_clean = Path('data/clean')
base_clean.mkdir(parents=True, exist_ok=True)

# 1) Cargar las dos últimas temporadas de jugadores argentinos
cur = pd.read_excel(base_raw / 'currentseason_argentinos.xlsx').assign(Season='Current')
prev = pd.read_excel(base_raw / 'previousseason_argentinos.xlsx').assign(Season='Previous')

all_seasons = pd.concat([cur, prev], ignore_index=True)

# Nos aseguramos de que los minutos sean numéricos
minutes_col = 'Minutes played'
all_seasons[minutes_col] = all_seasons[minutes_col].astype(float)

# Métricas por 90 minutos que usaremos en todo el análisis
metrics = [
    'Goals per 90',
    'xG per 90',
    'xA per 90',
    'Shots per 90',
    'Dribbles per 90',
    'Touches in box per 90',
    'Progressive passes per 90',
]

metrics = [m for m in metrics if m in all_seasons.columns]

for m in metrics:
    all_seasons[m] = all_seasons[m].astype(float)

# 2) Construir tabla agregada por jugador usando promedio ponderado por minutos
#    Además calculamos la versión total (sin normalizar) para cada métrica por 90
rows = []
for player, g in all_seasons.groupby('Player'):
    mins = g[minutes_col].sum()
    data = {
        'Player': player,
        minutes_col: mins,
    }
    for m in metrics:
        value_per90 = (g[m] * g[minutes_col]).sum() / mins if mins > 0 else float('nan')
        data[m] = value_per90
        total_raw = (g[m] * g[minutes_col]).sum() / 90 if mins > 0 else float('nan')
        data[f"{m} total"] = total_raw
    # xG+xA combinados
    if 'xG per 90' in metrics and 'xA per 90' in metrics:
        total_xg = (g['xG per 90'] * g[minutes_col]).sum() / 90 if mins > 0 else float('nan')
        total_xa = (g['xA per 90'] * g[minutes_col]).sum() / 90 if mins > 0 else float('nan')
        total_xgxa = total_xg + total_xa
        per90_xgxa = total_xgxa * 90 / mins if mins > 0 else float('nan')
        data['xG+xA per 90'] = per90_xgxa
        data['xG+xA total'] = total_xgxa
    rows.append(data)

agg = pd.DataFrame(rows)

# Añadimos el equipo: priorizamos el de la temporada actual si existe
team_current = cur.set_index('Player')['Team']
team_any = all_seasons.set_index('Player')['Team']
agg['Team'] = agg['Player'].map(lambda p: team_current.get(p, team_any.get(p)))

# Reordenamos columnas: primero Player/Team/minutos, luego métricas por 90, luego totales
per90_cols = metrics + (['xG+xA per 90'] if 'xG per 90' in metrics and 'xA per 90' in metrics else [])
total_cols = [f"{m} total" for m in metrics]
if 'xG per 90' in metrics and 'xA per 90' in metrics:
    total_cols.append('xG+xA total')
cols = ['Player', 'Team', minutes_col] + per90_cols + total_cols
agg = agg[cols]

# Guardar versión de depuración (opcional)
agg.to_excel(base_clean / 'argentinian_agg_debug.xlsx', index=False)

# 3) Extraer el perfil combinado de Lautaro para Tableau
lautaro_mask = agg['Player'] == 'L. Martínez'
lautaro_row = agg[lautaro_mask]
lautaro_row.to_excel(base_clean / 'df_lautaro_tableau.xlsx', index=False)

# 4) Calcular top 10 de candidatos similares a Lautaro (solo con métricas por 90)
num_cols = metrics
players_num = agg.copy()

lautaro_vec = (
    players_num.loc[players_num['Player'] == 'L. Martínez', num_cols]
    .iloc[0]
    .astype(float)
)

vals = players_num[num_cols].astype(float)

# Normalización tipo z-score
vals_norm = (vals - vals.mean()) / vals.std(ddof=0)
lautaro_norm = (lautaro_vec - vals.mean()) / vals.std(ddof=0)

dists = norm(vals_norm.values - lautaro_norm.values, axis=1)
players_num['distance_to_lautaro'] = dists

# Penalización por pocos minutos (también combinados)
minutes = players_num[minutes_col].astype(float)
minutes_norm = (minutes.max() - minutes) / (minutes.max() - minutes.min() + 1e-9)

# Normalizar distancias al rango [0, 1]
dist_norm = (players_num['distance_to_lautaro'] - players_num['distance_to_lautaro'].min()) / (
    players_num['distance_to_lautaro'].max() - players_num['distance_to_lautaro'].min() + 1e-9
)

players_num['replacement_score'] = 0.7 * dist_norm + 0.3 * minutes_norm

# Excluir a Lautaro de los candidatos
candidates = players_num[players_num['Player'] != 'L. Martínez']

cols_top10 = ['Player', 'Team', minutes_col, 'distance_to_lautaro', 'replacement_score']

candidates = candidates.sort_values('replacement_score', ascending=True)
df_top10 = candidates[cols_top10].head(10).reset_index(drop=True)
df_top10.insert(0, 'Rank', df_top10.index + 1)

df_top10.to_excel(base_clean / 'df_top10_lautaro.xlsx', index=False)

# 5) Perfil de percentiles de Lautaro con las métricas combinadas (por 90)
lautaro_agg = lautaro_row.iloc[0]
rows_profile = []
for m in metrics:
    series = agg[m].astype(float)
    val = float(lautaro_agg[m])
    pct = percentileofscore(series, val, kind='rank')
    rows_profile.append({'player': lautaro_agg['Player'], 'Metric': m, 'Percentile': pct})

metric_map = {
    'Goals per 90': 'Goals / 90',
    'xG per 90': 'xG / 90',
    'xA per 90': 'xA / 90',
    'Shots per 90': 'Shots / 90',
    'Dribbles per 90': 'Dribbles / 90',
    'Touches in box per 90': 'Touches in box / 90',
    'Progressive passes per 90': 'Progressive passes / 90',
}

df_profile = pd.DataFrame(rows_profile)
df_profile['Metric'] = df_profile['Metric'].map(metric_map)

df_profile.to_excel(base_clean / 'lautaro_percentile_profile.xlsx', index=False)

agg.head(), lautaro_row, df_top10, df_profile
